# Community Detection on Graphs

---

## Overview

**Community detection** finds groups of nodes in a graph that are more densely connected to each other than to the rest of the network.

**Modularity** measures the quality of a partition $\{C_1, \dots, C_k\}$:

$$Q = \frac{1}{2m} \sum_{u,v} \left[ A_{uv} - \frac{k_u k_v}{2m} \right] \delta(c_u, c_v)$$

where $m$ = number of edges, $k_u$ = degree of node $u$, $\delta(c_u, c_v) = 1$ if $u$ and $v$ are in the same community.

**Greedy Modularity Maximization** (Clauset-Newman-Moore):
1. Start with each node in its own community
2. Merge communities that give the largest gain in $Q$
3. Repeat until no merge improves $Q$

---

**Dataset:** Karate Club Graph (`nx.karate_club_graph()`). no download needed  
**Task:** Detect social communities in Zachary's karate club network.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import seaborn as sns

sns.set_theme()

from rice_ml.unsupervised_learning import CommunityDetector

In [ ]:
# Karate club graph. classic benchmark for community detection
G = nx.karate_club_graph()

print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print(f'Average degree: {np.mean([d for _, d in G.degree()]):.2f}')

In [ ]:
# Visualize the raw graph
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos, node_size=300, node_color='steelblue',
                 font_size=9, font_color='white', edge_color='gray')
plt.title('Karate Club Graph', fontsize=18)
plt.axis('off')
plt.show()

## Detect Communities. Greedy Modularity

In [ ]:
detector = CommunityDetector(method='greedy')
detector.fit(G)

print(f'Communities found: {len(detector.communities_)}')
for i, comm in enumerate(detector.communities_):
    print(f'  Community {i}: {sorted(comm)}')

In [ ]:
from networkx.algorithms.community.quality import modularity

Q = modularity(G, detector.communities_)
print(f'Number of communities: {len(detector.communities_)}')
print(f'Modularity Q:          {Q:.4f}')
print('(Q > 0.3 indicates meaningful community structure)')
for i, comm in enumerate(detector.communities_):
    print(f'  Community {i}: {len(comm)} nodes')


In [ ]:
# Color nodes by community
palette = ['red', 'lightseagreen', 'steelblue', 'magenta', 'orange']
node_colors = [palette[detector.labels_[node] % len(palette)] for node in G.nodes()]

plt.figure(figsize=(12, 8))
nx.draw_networkx(G, pos,
                 node_color=node_colors,
                 node_size=400,
                 font_size=9,
                 font_color='white',
                 edge_color='gray')
plt.title(f'Community Detection: {len(detector.communities_)} Communities', fontsize=18)
plt.axis('off')
plt.show()

In [ ]:
# Compare with known ground-truth club memberships
true_labels = [G.nodes[n]['club'] for n in G.nodes()]
true_colors = ['gold' if lbl == 'Mr. Hi' else 'mediumpurple' for lbl in true_labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

nx.draw_networkx(G, pos, node_color=node_colors, node_size=300,
                 font_size=8, font_color='white', edge_color='gray', ax=ax1)
ax1.set_title('Detected Communities', fontsize=15)
ax1.axis('off')

nx.draw_networkx(G, pos, node_color=true_colors, node_size=300,
                 font_size=8, font_color='white', edge_color='gray', ax=ax2)
ax2.set_title('True Club Memberships', fontsize=15)
ax2.axis('off')

plt.suptitle('Community Detection vs Ground Truth', fontsize=18)
plt.tight_layout()
plt.show()

## Interpretation

- The karate club split into two factions after an instructor-student conflict. community detection recovers this split.
- **Modularity** measures how much more densely connected communities are internally vs. at random.
- Community detection is used in social network analysis, biology (protein interaction networks), and recommendation systems.